# TALLER 2: Limpieza de Datos - Del Caos al Orden

## Descripción
Los datos del mundo real nunca vienen limpios. En este taller aprenderemos a detectar y corregir problemas comunes: valores faltantes, duplicados, tipos incorrectos y outliers. Sin este paso, los modelos de ML dan resultados incorrectos.

## Regla de oro de la limpieza
"Garbage in, Garbage out" — Un modelo es tan bueno como los datos con los que aprende. Datos sucios = predicciones incorrectas.

## Problemas que aprenderemos a resolver:
- **NaN** — Valores faltantes (el sensor falló, el usuario no respondió)
- **Duplicados** — El mismo registro dos veces sesga el modelo
- **Tipos incorrectos** — Columnas guardadas como texto impedirán operaciones matemáticas
- **Outliers** — Valores extremos que distorsionan la distribución

In [ ]:
# IMPORTANTE: ejecuta este bloque antes de los ejercicios
import pandas as pd
import numpy as np

# Cargar el dataset
df = pd.read_csv('ventas.csv', parse_dates=['fecha'])

# Crear una copia para trabajar (siempre es buena práctica)
df_clean = df.copy()

# Ver el estado inicial
print('=' * 60)
print('ESTADO INICIAL DEL DATASET')
print('=' * 60)
print(f'Shape: {df_clean.shape}')
print()
print('Nulos por columna:')
print(df_clean.isnull().sum())
print()
print(f'Duplicados: {df_clean.duplicated().sum()}')

## Ejercicio 1: Detectar valores faltantes (NaN)

**Objetivo:** Identificar la cantidad y porcentaje de valores faltantes en cada columna.

**Explicación:**
Los valores NaN (Not a Number) representan datos faltantes. Es crucial saber:
- Cuántos NaN tiene cada columna
- Qué porcentaje del total representa
- Si el patrón de NaN tiene sentido (datos faltantes correlacionados)

In [ ]:
# Ejercicio 1: Análisis completo de valores faltantes
print('=' * 60)
print('ANALISIS DE VALORES FALTANTES')
print('=' * 60)

# Cantidad de NaN por columna
nulos_por_columna = df_clean.isnull().sum()
print('Cantidad de NaN por columna:')
print(nulos_por_columna)
print()

# Porcentaje de NaN por columna
porcentaje_nulos = (df_clean.isnull().sum() / len(df_clean)) * 100
print('Porcentaje de NaN por columna:')
print(porcentaje_nulos.round(2))
print()

# Columnas con valores faltantes
columnas_con_nulos = nulos_por_columna[nulos_por_columna > 0]
print(f'Columnas con valores faltantes: {len(columnas_con_nulos)}')
print(list(columnas_con_nulos.index))

## Ejercicio 2: Eliminar valores faltantes (dropna)

**Objetivo:** Aprender cuándo y cómo eliminar filas o columnas con valores faltantes.

**Explicación:**
Cuándo eliminar:
- Más del 60% de la columna es NaN
- Las filas con NaN son < 5% del total

Pros: Simple, rápido. No introduce valores artificiales.
Contras: Pierdes datos reales.

In [ ]:
# Ejercicio 2: Eliminar valores faltantes
print('=' * 60)
print('ELIMINAR VALORES FALTANTES')
print('=' * 60)

# Verificar estado actual
print(f'Shape inicial: {df_clean.shape}')
print(f'NaN en ciudad: {df_clean["ciudad"].isnull().sum()}')
print()

# Eliminar filas con NaN en columnas específicas (subset)
df_sin_nulos_ciudad = df_clean.dropna(subset=['ciudad'])
print(f'Despues de eliminar NaN en ciudad: {df_sin_nulos_ciudad.shape}')
print(f'Filas eliminadas: {df_clean.shape[0] - df_sin_nulos_ciudad.shape[0]}')
print()

# Eliminar columnas con mas del 60% de NaN
umbral = 0.6
columnas_a_eliminar = df_clean.columns[df_clean.isnull().mean() > umbral]
print(f'Columnas con mas de {umbral*100}% de NaN: {list(columnas_a_eliminar)}')

## Ejercicio 3: Imputar valores faltantes (fillna)

**Objetivo:** Llenar valores faltantes con estadisticos o valores especificos.

**Explicación:**
Estrategias segun el tipo de dato:
- **Numerico sin outliers** → Media (mean)
- **Numerico con outliers** → Mediana (median)
- **Categorico** → Moda (valor mas frecuente)

In [ ]:
# Ejercicio 3: Imputar valores faltantes
print('=' * 60)
print('IMPUTAR VALORES FALTANTES')
print('=' * 60)

# Volver a cargar para demostrar diferentes estrategias
df_demo = df.copy()

# Estrategia 1: Imputar 'ciudad' con la moda (valor mas frecuente)
ciudad_moda = df_demo['ciudad'].mode()[0]
print(f'Moda de ciudad: {ciudad_moda}')
df_demo['ciudad'] = df_demo['ciudad'].fillna(ciudad_moda)
print(f'NaN en ciudad despues de imputar: {df_demo["ciudad"].isnull().sum()}')
print()

# Estrategia 2: Imputar 'items' con la mediana (resiste outliers)
items_mediana = df_demo['items'].median()
print(f'Mediana de items: {items_mediana}')
df_demo['items'] = df_demo['items'].fillna(items_mediana)
print(f'NaN en items despues de imputar: {df_demo["items"].isnull().sum()}')
print()

# Estrategia 3: Imputar 'costo' con la mediana
costo_mediana = df_demo['costo'].median()
print(f'Mediana de costo: {costo_mediana:.2f}')
df_demo['costo'] = df_demo['costo'].fillna(costo_mediana)
print(f'NaN en costo despues de imputar: {df_demo["costo"].isnull().sum()}')

## Ejercicio 4: Detectar y eliminar duplicados

**Objetivo:** Identificar y eliminar registros duplicados que sesgan el modelo.

**Explicación:**
Los duplicados dan "peso" artificial a ciertos datos, sesgando el modelo.
Usamos `duplicated()` para detectar y `drop_duplicates()` para eliminar.

In [ ]:
# Ejercicio 4: Detectar duplicados
print('=' * 60)
print('DETECTAR DUPLICADOS')
print('=' * 60)

# Verificar si hay duplicados exactos en todo el DataFrame
duplicados_totales = df_clean.duplicated().sum()
print(f'Duplicados exactos en todo el DataFrame: {duplicados_totales}')
print()

# Verificar duplicados en columnas especificas
duplicados_pedido = df_clean.duplicated(subset=['pedido_id']).sum()
print(f'Duplicados por pedido_id: {duplicados_pedido}')

In [ ]:
# Ejercicio 4: Eliminar duplicados
print('=' * 60)
print('ELIMINAR DUPLICADOS')
print('=' * 60)

print(f'Shape antes de eliminar duplicados: {df_clean.shape}')

# Eliminar duplicados basados en columna especifica
# Mantener el primer registro de cada pedido_id
df_sin_duplicados = df_clean.drop_duplicates(subset=['pedido_id'], keep='first')
print(f'Shape despues de eliminar duplicados por pedido_id: {df_sin_duplicados.shape}')
print(f'Registros eliminados: {df_clean.shape[0] - df_sin_duplicados.shape[0]}')

## Ejercicio 5: Corregir tipos de datos

**Objetivo:** Asegurar que cada columna tenga el tipo de dato correcto.

**Explicación:**
Los tipos incorrectos impiden operaciones matematicas (ej: precio como texto).
Verificamos con `dtypes` y convertimos con `astype()` o `pd.to_numeric()`.

In [ ]:
# Ejercicio 5: Verificar tipos de datos
print('=' * 60)
print('TIPOS DE DATOS ACTUALES')
print('=' * 60)

print(df_clean.dtypes)

In [ ]:
# Ejercicio 5: Convertir tipos de datos
print('=' * 60)
print('CONVERTIR TIPOS DE DATOS')
print('=' * 60)

# Asegurar que 'fecha' sea datetime
df_clean['fecha'] = pd.to_datetime(df_clean['fecha'])
print(f"fecha convertida a: {df_clean['fecha'].dtype}")
print()

# Convertir columnas numericas si estan como texto
for col in ['precio', 'monto', 'costo', 'items']:
    try:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f'{col} convertida a: {df_clean[col].dtype}')
    except Exception as e:
        print(f'Error con {col}: {e}')

## Ejercicio 6: Detectar y manejar outliers (IQR)

**Objetivo:** Identificar valores extremos usando el metodo del Rango Intercuartilico (IQR).

**Explicación:**
El metodo IQR define limites para detectar outliers:
- IQR = Q3 - Q1
- Limite inferior = Q1 - 1.5 × IQR
- Limite superior = Q3 + 1.5 × IQR
- Valores fuera de estos limites = OUTLIERS

El factor 1.5 captura el 99.3% de una distribucion normal.

In [ ]:
# Ejercicio 6: Detectar outliers con IQR
print('=' * 60)
print('DETECTAR OUTLIERS CON IQR')
print('=' * 60)

def detectar_outliers_iqr(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    
    print(f'Columna: {columna}')
    print(f'  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}')
    print(f'  Limite inferior: {limite_inferior:.2f}')
    print(f'  Limite superior: {limite_superior:.2f}')
    print(f'  Outliers encontrados: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')
    print()
    
    return outliers

# Detectar outliers en columnas numericas
for col in ['monto', 'precio', 'items']:
    outliers = detectar_outliers_iqr(df_clean, col)

In [ ]:
# Ejercicio 6: Visualizar outliers con boxplot
import matplotlib.pyplot as plt

print('=' * 60)
print('VISUALIZAR OUTLIERS CON BOXPLOT')
print('=' * 60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(['monto', 'precio', 'items']):
    axes[i].boxplot(df_clean[col].dropna())
    axes[i].set_title(f'Boxplot de {col}')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Ejercicio 6: Manejar outliers (opciones)
print('=' * 60)
print('OPCIONES PARA MANEJAR OUTLIERS')
print('=' * 60)

# Opcion 1: Eliminar outliers
def eliminar_outliers_iqr(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return df[(df[columna] >= limite_inferior) & (df[columna] <= limite_superior)]

# Ejemplo: eliminar outliers de 'monto'
print(f'Shape antes: {df_clean.shape}')
df_sin_outliers = eliminar_outliers_iqr(df_clean, 'monto')
print(f'Shape despues de eliminar outliers en monto: {df_sin_outliers.shape}')
print()

# Opcion 2: Winsorizar (capped) - reemplazar outliers por los limites
def winsorize_iqr(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    df[columna] = df[columna].clip(lower=limite_inferior, upper=limite_superior)
    return df

print('Nota: Tambien puedes winsorizar (limitar) los outliers en lugar de eliminarlos')

## Ejercicio 7: Pipeline completo de limpieza

**Objetivo:** Aplicar todas las tecnicas de limpieza en un pipeline ordenado.

In [ ]:
# Ejercicio 7: Pipeline completo de limpieza
print('=' * 60)
print('PIPELINE COMPLETO DE LIMPIEZA')
print('=' * 60)

# Volver a cargar datos originales
df_final = df.copy()

# Estado inicial
print('ESTADO INICIAL:')
print(f'  Shape: {df_final.shape}')
print(f'  Nulos: {df_final.isnull().sum().sum()}')
print(f'  Duplicados: {df_final.duplicated().sum()}')
print()

# Paso 1: Eliminar duplicados por pedido_id
df_final = df_final.drop_duplicates(subset=['pedido_id'], keep='first')
print('Paso 1: Duplicados eliminados')
print(f'  Shape: {df_final.shape}')
print()

# Paso 2: Convertir tipos de datos
df_final['fecha'] = pd.to_datetime(df_final['fecha'])
for col in ['precio', 'monto', 'costo', 'items']:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
print('Paso 2: Tipos de datos corregidos')
print()

# Paso 3: Imputar valores faltantes
# Ciudad: imputar con moda
df_final['ciudad'] = df_final['ciudad'].fillna(df_final['ciudad'].mode()[0])
# Items: imputar con mediana
df_final['items'] = df_final['items'].fillna(df_final['items'].median())
# Costo: imputar con mediana
df_final['costo'] = df_final['costo'].fillna(df_final['costo'].median())
print('Paso 3: Valores faltantes imputados')
print(f'  Nulos restantes: {df_final.isnull().sum().sum()}')
print()

# Paso 4: Verificar outliers (solo mostrar)
print('Paso 4: Verificacion de outliers')
Q1 = df_final['monto'].quantile(0.25)
Q3 = df_final['monto'].quantile(0.75)
IQR = Q3 - Q1
outliers_monto = df_final[(df_final['monto'] < Q1 - 1.5*IQR) | (df_final['monto'] > Q3 + 1.5*IQR)]
print(f'  Outliers en monto: {len(outliers_monto)}')
print()

# Estado final
print('=' * 60)
print('ESTADO FINAL:')
print(f'  Shape: {df_final.shape}')
print(f'  Nulos: {df_final.isnull().sum().sum()}')
print(f'  Duplicados: {df_final.duplicated().sum()}')

In [ ]:
# Ejercicio 7: Generar reporte de calidad ANTES vs DESPUES
print('=' * 60)
print('REPORTE DE CALIDAD: ANTES vs DESPUES')
print('=' * 60)

# Cargar datos originales para comparacion
df_original = pd.read_csv('ventas.csv', parse_dates=['fecha'])

reporte = pd.DataFrame({
    'Metrica': ['Filas', 'Columnas', 'Valores Nulos', 'Duplicados'],
    'ANTES': [
        df_original.shape[0],
        df_original.shape[1],
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum()
    ],
    'DESPUES': [
        df_final.shape[0],
        df_final.shape[1],
        df_final.isnull().sum().sum(),
        df_final.duplicated().sum()
    ]
})

reporte['Diferencia'] = reporte['ANTES'] - reporte['DESPUES']
print(reporte.to_string(index=False))

## Errores comunes a evitar

### 1. Data Leakage (el mas grave)
- ✗ Escalar X_train + X_test juntos ANTES de dividir
- ✓ Dividir primero → fit_transform en X_train → transform (solo) en X_test

### 2. SettingWithCopyWarning
- ✗ df_sub = df[mask] → df_sub['col'] = valor
- ✓ df_sub = df[mask].copy() → df_sub es independiente

### 3. Imputar antes de dividir train/test
- ✗ Calcular la media de TODO el dataset e imputar
- ✓ Dividir → calcular media SOLO de X_train → imputar ambos con ese valor

## Resumen del Taller 2

En este taller cubrimos las tecnicas fundamentales de limpieza de datos:

1. **Deteccion de NaN:** `isnull().sum()`, porcentaje de valores faltantes
2. **Eliminacion de NaN:** `dropna()` cuando hay pocos datos faltantes
3. **Imputacion:** `fillna()` con media, mediana o moda
4. **Duplicados:** `duplicated()` y `drop_duplicates()`
5. **Tipos de datos:** `pd.to_datetime()`, `pd.to_numeric()`
6. **Outliers:** Metodo IQR para detectar y manejar valores extremos

Un dataset limpio es esencial para que los modelos de Machine Learning funcionen correctamente.